In [1]:
from ovo import db, models_proteinqc, project_logic, descriptor_logic, schedulers, storage
import os
import glob

Registering plugin ovo_promb
Registering plugin ovo_proteindj


OVO home /home/username/ovo

In [2]:
project, project_round = project_logic.get_or_create_project_round("OVO Publication Examples 1", "Jain Shehata antibodies")

In [8]:
username = os.getenv('USER')
print(username)

username


In [10]:
# Create pool
pool = db.Pool(
    id=db.Pool.generate_id(),
    author=username,
    round_id=project_round.id,
    name="Jain Shehata",
    description="ImmuneBuilder raw numbering antibodies from Jain & Shehata"
)
db.save(pool)
pool.id

'efm'

In [ ]:
%%bash
    
if [ ! -d ../../data/jain_shehata/immunebuilder_raw_numbering/ ]; then
    cd ../../data/jain_shehata/; unzip immunebuilder_raw_numbering.zip
fi

In [5]:
paths = glob.glob('../../data/jain_shehata/immunebuilder_raw_numbering/*.pdb')
len(paths)

537

In [11]:
designs = []
for path in paths:
    with open(path) as f:
        pdb_str = f.read()
    designs.append(db.Design.from_pdb_file(
        storage=storage,
        filename=os.path.basename(path),
        pdb_str=pdb_str,
        chains=['H', 'L'],
        project_id=project.id,
        pool_id=pool.id,
    ))

# Save designs to db because we load them from db when submitting the proteinqc job
db.save_all(designs + [pool])

In [4]:
#pool = db.Pool.get(name="Jain Shehata")
#designs = db.Design.select(pool_id=pool.id)

In [5]:
workflow = models_proteinqc.ProteinQCWorkflow(
    chains=['H', 'L'],
    design_ids=[design.id for design in designs],
    tools=['peppatch', 'esm_1v', 'dssp', 'proteinsol', 'esm_if', 'seq_composition'],
)
workflow.validate()
workflow.get_table_row()

Workflow  type    ProteinQC workflow
dtype: object

In [7]:
print(schedulers.keys())

SCHEDULER_KEY = 'slurm_singularity'

dict_keys(['slurm_singularity', 'pbs_singularity', 'local_singularity', 'local_conda', 'local_single_gpu'])


In [8]:
#
# SUBMIT - note that running this multiple times will submit a the workflow each time!
#
descriptor_job = descriptor_logic.submit_descriptor_workflow(
    workflow=workflow,
    scheduler_key=SCHEDULER_KEY,
    project_id=project.id
)
descriptor_job.id

Submitting workflow: nextflow run -with-trace trace.txt -work-dir /home/username/ovo/workdir/work /home/username/projects/ovo-open-source/ovo/ovo/pipelines/proteinqc --publish_dir output --reference_files_dir /home/username/ovo/reference_files --shared_modules ovo:/home/username/projects/ovo-open-source/ovo/ovo,ovo_promb:/home/username/projects/ovo-open-source/ovo-promb/ovo_promb,ovo_proteindj:/home/username/projects/ovo-proteindj/ovo_proteindj -config /home/username/projects/ovo-open-source/ovo/ovo/pipelines/nextflow_default.config -config /home/username/projects/ovo-open-source/ovo/ovo/pipelines/proteinqc/nextflow.config -profile singularity -config /home/username/ovo/nextflow_slurm_singularity.config -ansi-log false -bg --input_pdb /home/username/ovo/workdir/inputs/a8/1cdba67a7daa79faec4e98baf9b3072fdda8ca/input_pdb_paths.txt --tools esm_1v,dssp,proteinsol,esm_if,seq_composition --chains H,L --batch_size 50
Execution directory: /home/username/ovo/workdir/execdir/da546eca-c175-11f0-9

'73d573'

In [9]:
descriptor_logic.process_results(descriptor_job)

Waiting for job da546eca-c175-11f0-9331-0248d8152725 to finish...


In [10]:
db.DescriptorValue.count(design_id__in=db.Design.select_values('id', pool_id=pool.id))

50478

In [11]:
values = descriptor_logic.get_wide_descriptor_table(
    design_ids=db.Design.select_values('id', pool_id=pool.id),
)
print(len(values))
values.head()

537


,Sequence L,Sequence H,Sequence length,Ala %,Cys %,Asp %,Glu %,Phe %,Gly %,His %,...,Normalized positive top1 patch area at pH 7.4,Negative patch area at pH 7.4,Normalized negative patch area at pH 7.4,Negative top1 patch area at pH 7.4,Normalized negative top1 patch area at pH 7.4,Electrostatic volume integral at pH 7.4,Positive electrostatic regions volume integral at pH 7.4,Negative electrostatic regions volume integral at pH 7.4,Positive electrostatic volume integral at pH 7.4,Negative electrostatic volume integral at pH 7.4
design_id,,,,,,,,,,,,,,,,,,,,,
ovo_efm_ADI-47123,DIQLTQSPSSLSASVGDSVAITCRATQDINSFLAWYQQEPGKAPKL...,QVQLQQWGAGLLKPSETLSLTCAVSGGSLNNYYWTWIRQPPGKGLE...,229,6.550218,1.746725,3.930131,2.620087,3.056769,8.733624,0.436681,...,0.011880,29.662641,0.003030,11.049521,0.001129,7283.738623,6274.675184,-2315.091245,17119.411311,-9835.672688
ovo_efm_ADI-47204,SYVLTQPPSVSVAPGQTARITCGGNNIGSKRVHWYRQRPGQAPVLV...,EVQLLESGGGVVQPGRSLRLSCVVSGFMFSGYGMHWVRRAPGKGLE...,233,6.437768,1.716738,5.579399,2.575107,2.575107,12.017167,1.287554,...,0.013006,228.988793,0.022559,97.764718,0.009631,4749.533649,4388.045839,-3640.496343,14962.895164,-10213.361515
ovo_efm_ADI-45422,EIVMTQSPGTLSLSPGERATLSCRASQSVSSSYLAWYQQKPGQAPR...,QVQLVESGGGVVQPGRSLRLSCAASGFTFSSYGMHWVRQAPGKGLE...,231,6.926407,1.731602,3.463203,3.463203,3.030303,9.956710,0.432900,...,0.012732,225.457774,0.022271,151.804523,0.014995,19982.949347,8780.981053,-2165.181603,25662.907849,-5679.958502
ovo_efm_benralizumab,DIQMTQSPSSLSASVGDRVTITCGTSEDIINYLNWYQQKPGKAPKL...,EVQLVQSGAEVKKPGASVKVSCKASGYTFTSYVIHWVRQRPGQGLA...,228,3.508772,1.754386,3.947368,3.947368,2.631579,10.526316,0.877193,...,0.029318,103.946762,0.010119,58.742999,0.005719,23027.179664,14115.362119,-3001.367834,31042.644924,-8015.465260
ovo_efm_crenezumab,DIVMTQSPLSLPVTPGEPASISCRSSQSLVYSNGDTYLHWYLQKPG...,EVQLVESGGGLVQPGGSLRLSCAASGFTFSSYGMSWVRQAPGKGLE...,224,4.464286,1.785714,4.017857,3.571429,3.125000,11.160714,0.892857,...,0.000518,256.597374,0.026048,83.926587,0.008520,7.107200,3080.033527,-4345.613991,12534.448962,-12527.341762


In [12]:
values.to_csv('../../data/results/06_proteinqc_jain_shehata_antibodies/descriptors.csv')